In [1]:
from utils import *
import numpy as np
import torch
from torch.nn.functional import silu

In [2]:
class PixelsDataset():
    def __init__(self, path, split):
        Z, _, _ = load_invert(path)
        N, L, C, H, W = Z.shape
        N = int(N*0.9)
        if split == 'train':
            Z = Z[:N ,...]
        elif split == 'val':
            Z = Z[N: ,...]

        self.size = Z.shape[0] * (L - 1) * C * H * W
        self.H = H
        self.W = W
        self.C = C
        self.L = L

        self.Z_t = torch.from_numpy(Z)
        self.xs = torch.arange(H, dtype=torch.float32)
        self.ys = torch.arange(W, dtype=torch.float32)
        self.leads = (torch.arange(2, L+1, dtype=torch.float32) -2) / (L - 2)  # leads 2,...,24

        batch_numbers = (L-1) * 15
        self.batch_size = self.size // batch_numbers

    def iterate_batches(self):
        # int32 and no .tolist(): keeps an exact shuffle at 4 bytes per index
        perm = torch.randperm(self.size, dtype=torch.int32)
        for start in range(0, self.size, self.batch_size):
            idx = perm[start:start + self.batch_size]
            y = idx % self.W
            idx = idx // self.W
            x = idx % self.H
            idx = idx // self.H
            ch = idx % self.C
            idx = idx // self.C
            t = idx % (self.L - 1)
            n = idx // (self.L - 1)
            # Z index t+1 is lead t+2, which is what leads[t] encodes
            yield self.leads[t], ch, self.xs[x], self.ys[y], self.Z_t[n, t, ch, x, y], self.Z_t[n, t + 1, ch, x, y]

train_ds = PixelsDataset(path = ROOT.parent / 'data' / 'invert' / 'from_val_rk4', split='train')
# test code
lead, ch, x, y, z_prev, z_next = next(train_ds.iterate_batches())
print(int(train_ds.size/(0.8*23*2*64*64))) 
print(lead.shape, ch.shape, x.shape, y.shape, z_prev.shape, z_next.shape)
print(f"x max: {torch.max(x)}, x min {torch.min(x)}")
print(f"y max: {torch.max(y)}, y min {torch.min(y)}")
print(f"lead max: {torch.max(lead)}, lead min {torch.min(lead)}")
        

168
torch.Size([73728]) torch.Size([73728]) torch.Size([73728]) torch.Size([73728]) torch.Size([73728]) torch.Size([73728])
x max: 63.0, x min 0.0
y max: 63.0, y min 0.0
lead max: 1.0, lead min 0.0


In [3]:
val_ds = PixelsDataset(path = ROOT.parent / 'data' / 'invert' / 'from_val_rk4', split='val')
print(int(val_ds.size/(0.2*23*2*64*64))) 
print(val_ds.batch_size)

74
8192


In [4]:
class PositionalEmbedding(torch.nn.Module):
    def __init__(self, num_channels, max_positions=24, endpoint=False):
        super().__init__()
        self.num_channels = num_channels
        self.max_positions = max_positions
        self.endpoint = endpoint

    def forward(self, x):
        #FROMTRACE:
        # num_channels decide how long the output will be
        # max_position decides the slowest wave
        
        # Define the frequencies domain from [0, num_channels//2]
        # Normalise the frequencies domain to [0, 1]
        # For each f: x = x * 10000^(-f)
        freqs = torch.arange(start=0, end=self.num_channels //
                             2, dtype=torch.float32, device=x.device)
        freqs = freqs / (self.num_channels // 2 - (1 if self.endpoint else 0))
        freqs = (1 / self.max_positions) ** freqs
        x = x.ger(freqs.to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

class FourierEmbedding(torch.nn.Module):
    def __init__(self, num_channels, period=64):
        super().__init__()
        n_freq = num_channels // 2                    # F = num_channels / 2
        freqs = torch.arange(1, n_freq + 1, dtype=torch.float32) / period
        self.register_buffer('freqs', freqs)   

    def forward(self, x):
        # Outer product
        x = x.ger((2 * np.pi * self.freqs).to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

class TimeVaryingOU(torch.nn.Module):
    # embedding t with positional embedding
    # embedding x with fourier embedding
    # embedding y with fourier embedding
    # embedding ch with nn.Embedding(2, d)
    # 3-4 MLP layers with SiLU
    # end up with softplus 
    def __init__(self, emb_dim=64, hidden_dim=256, n_hidden=3):
        super().__init__()
        self.t_emb = PositionalEmbedding(emb_dim)
        self.x_emb = FourierEmbedding(emb_dim)
        self.y_emb = FourierEmbedding(emb_dim)
        self.ch_emb = torch.nn.Embedding(2, emb_dim)

        dims = [4 * emb_dim] + [hidden_dim] * n_hidden
        self.layers = torch.nn.ModuleList(
            torch.nn.Linear(d_in, d_out) for d_in, d_out in zip(dims[:-1], dims[1:])
        )
        self.out = torch.nn.Linear(hidden_dim, 1)

    def forward(self, t, x, y, ch):
        embedded_x = self.x_emb(x)
        embedded_y = self.y_emb(y)
        embedded_ch = self.ch_emb(ch)
        embedded_t = self.t_emb(t)
        h = torch.cat([embedded_t, embedded_x, embedded_y, embedded_ch], dim=1)
        for layer in self.layers:
            h = silu(layer(h))
        return torch.nn.functional.softplus(self.out(h)).squeeze(-1)  # (B,)


In [5]:
def batch_loss(batch):
    lead, ch, x, y, z_prev, z_next = (b.to(device) for b in batch)
    c = model(lead, x, y, ch)
    alpha = torch.exp(-0.5 * c ** 2)
    loss = 0.5 * (torch.log(1 - alpha**2) + (z_next - alpha * z_prev)**2 / (1-alpha**2))
    return loss.mean(), len(lead)

In [6]:
def train(model, train_ds, val_ds, device, epochs=50, lr=1e-3):
    # dt is the gap between consecutive leads in hours, so c is in physical units
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {'train': [], 'val': []}
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for batch in train_ds.iterate_batches():
            loss, n = batch_loss(batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * n
        history['train'].append(total / train_ds.size)

        model.eval()
        total = 0.0
        with torch.no_grad():
            for batch in val_ds.iterate_batches():
                loss, n = batch_loss(batch)
                total += loss.item() * n
        history['val'].append(total / val_ds.size)

        print(f"epoch {epoch + 1}/{epochs}  train nll {history['train'][-1]:.4f}  val nll {history['val'][-1]:.4f}")
    return history


model = TimeVaryingOU()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
history = train(model, train_ds, val_ds, device)

epoch 1/50  train nll 0.0987  val nll 0.1005
epoch 2/50  train nll 0.0983  val nll 0.1005
epoch 3/50  train nll 0.0981  val nll 0.1006
epoch 4/50  train nll 0.0980  val nll 0.1005
epoch 5/50  train nll 0.0979  val nll 0.1007
epoch 6/50  train nll 0.0978  val nll 0.1008
epoch 7/50  train nll 0.0977  val nll 0.1007
epoch 8/50  train nll 0.0977  val nll 0.1007
epoch 9/50  train nll 0.0976  val nll 0.1007
epoch 10/50  train nll 0.0976  val nll 0.1006
epoch 11/50  train nll 0.0975  val nll 0.1007
epoch 12/50  train nll 0.0975  val nll 0.1007
epoch 13/50  train nll 0.0975  val nll 0.1009
epoch 14/50  train nll 0.0974  val nll 0.1007
epoch 15/50  train nll 0.0974  val nll 0.1007
epoch 16/50  train nll 0.0974  val nll 0.1008
epoch 17/50  train nll 0.0974  val nll 0.1010
epoch 18/50  train nll 0.0973  val nll 0.1008
epoch 19/50  train nll 0.0973  val nll 0.1008
epoch 20/50  train nll 0.0973  val nll 0.1009
epoch 21/50  train nll 0.0973  val nll 0.1008
epoch 22/50  train nll 0.0972  val nll 0.10